In [2]:
"""
Enhanced DTS-ESN grid search for closed-loop prediction of the Rulkov map.

Base model from:
  Tanaka, Matsumori, Yoshida, Aihara,
  "Reservoir computing with diverse timescales for prediction of multiscale
   dynamics", Phys. Rev. Research 4, L032014 (2022).

Readout remains STANDARD RIDGE REGRESSION throughout.  All improvements are
architectural / protocol-level, not learning-rule changes.

----------------------------------------------------------------------
ENHANCEMENTS OVER THE PAPER'S BASELINE (all preserve ridge readout):

  (1) Delay-embedded input and feedback.
      u(t) = [x_t, x_{t-tau}, x_{t-2 tau}].  Takens-style reconstruction
      of the missing Rulkov y variable from fast x alone.

  (2) Squared reservoir features in readout.
      y_hat = W_out @ [x; x**2].  Ridge still closed-form.  Helps the
      readout distinguish spike/silent regimes.

  (3) Fast + slow readout branches.
      W_out_fast predicts x itself (main output).
      W_out_slow predicts EMA(x, tau_slow), a learned surrogate of the
      missing Rulkov y.  Its output is fed back via a separate
      W_fb_slow matrix with its own gain zeta_slow.  This gives the
      reservoir an explicit low-pass view of its own trajectory.

  (4) Spike-weighted ridge.
      Samples near true spike peaks (+/- 2 steps) get 4x weight.
      Still closed-form: solve (X^T diag(w) X + beta I) W = X^T diag(w) Y.

  (5) Hybrid-forcing fade at sync boundary.
      First 50 closed-loop steps use lambda-blended feedback
      (1 - lambda) * truth + lambda * prediction, with lambda ramping 0 -> 1.

  (6) Scheduled noise injection on training inputs.
      sigma decays linearly from sigma_hi to sigma_lo during training.

  (7) Train/test gap + long sync window + composite metric + divergence
      rejection across seeds.

----------------------------------------------------------------------
COMPOSITE SCORE (per seed, then mean):
    score = NRMSE_short(50 steps) + 0.5 * |log(n_pred_spikes / n_true_spikes)|
Lower is better.  Any seed diverging (NaN / |y| > 1e6) rejects the config.
"""

import itertools
import time
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..',)))
from lib.utils.reservoirpy import (
    fit_scaler, transform_array, inverse_transform_array
)


# ==========================================================
# SPIKE DETECTION (user-provided)
# ==========================================================
def count_spikes(signal, threshold=0.0):
    s = np.asarray(signal).ravel()
    is_peak = (s[1:-1] > s[:-2]) & (s[1:-1] > s[2:]) & (s[1:-1] > threshold)
    return int(np.sum(is_peak))


def spike_indices(signal, threshold=0.0):
    """Return indices of local maxima above threshold."""
    s = np.asarray(signal).ravel()
    is_peak = (s[1:-1] > s[:-2]) & (s[1:-1] > s[2:]) & (s[1:-1] > threshold)
    return np.where(is_peak)[0] + 1  # +1 because we sliced from index 1


# ==========================================================
# DTS-ESN with delay embedding, squared features, dual readout
# ==========================================================
class DTSESN:
    """
    Diverse-timescale ESN with:
      * delay-embedded input and feedback (tau, n_delays)
      * squared reservoir features in readout
      * fast readout (main output) and slow readout (EMA surrogate)
      * slow-output feedback path with its own gain zeta_slow
    Readout trained with weighted ridge regression.
    """

    def __init__(self, n_reservoir,
                 alpha_min, alpha_max=1.0,
                 spectral_radius=1.0,
                 input_scaling=0.8, fb_scaling=1.0,
                 fb_scaling_slow=0.5,
                 density=0.1, ridge=1e-3,
                 tau=5, n_delays=3,
                 tau_slow=100,
                 noise_hi=0.01, noise_lo=0.001,
                 spike_weight=4.0, spike_window=2,
                 seed=42):
        self.Nx = int(n_reservoir)
        self.alpha_min = alpha_min
        self.alpha_max = alpha_max
        self.rho = spectral_radius
        self.gamma = input_scaling
        self.zeta_fast = fb_scaling
        self.zeta_slow = fb_scaling_slow
        self.density = density
        self.ridge = ridge
        self.tau = int(tau)
        self.n_delays = int(n_delays)
        self.tau_slow = int(tau_slow)
        self.noise_hi = noise_hi
        self.noise_lo = noise_lo
        self.spike_weight = spike_weight
        self.spike_window = int(spike_window)
        self.rng = np.random.default_rng(seed)

        self._build_weights()
        self._build_leaks()
        self.W_out = None      # shape (2, 2*Nx) -- [fast; slow] from [x; x^2]
        self.x = np.zeros(self.Nx)

    # ---- construction ----
    def _build_weights(self):
        Nx = self.Nx
        W = self.rng.uniform(-1, 1, size=(Nx, Nx))
        mask = self.rng.uniform(0, 1, size=(Nx, Nx)) < self.density
        W = W * mask
        eig = np.linalg.eigvals(W)
        sr = float(np.max(np.abs(eig)))
        if sr > 0:
            W = W / sr
        self.W = self.rho * W
        # Delay-embedded input: n_delays scalars per step
        self.W_in = self.rng.uniform(-1, 1, size=(Nx, self.n_delays))
        # Fast-feedback: single scalar (the primary output)
        self.W_fb_fast = self.rng.uniform(-1, 1, size=(Nx, 1))
        # Slow-feedback: single scalar (the EMA surrogate)
        self.W_fb_slow = self.rng.uniform(-1, 1, size=(Nx, 1))

    def _build_leaks(self):
        if self.alpha_min == self.alpha_max:
            self.alphas = np.full(self.Nx, self.alpha_max)
        else:
            la = self.rng.uniform(np.log10(self.alpha_min),
                                  np.log10(self.alpha_max), self.Nx)
            self.alphas = 10.0 ** la

    # ---- core step ----
    def _features(self):
        """Readout features: [x; x^2]."""
        return np.concatenate([self.x, self.x * self.x])

    def _step(self, u_delayed, y_fast, y_slow):
        """
        u_delayed: (n_delays,) current + delayed inputs
        y_fast:    (1,) previous fast feedback
        y_slow:    (1,) previous slow feedback
        """
        h = (self.W @ self.x
             + self.gamma * (self.W_in @ u_delayed)
             + self.zeta_fast * (self.W_fb_fast @ np.atleast_1d(y_fast))
             + self.zeta_slow * (self.W_fb_slow @ np.atleast_1d(y_slow)))
        self.x = (1.0 - self.alphas) * self.x + self.alphas * np.tanh(h)

    def reset(self):
        self.x = np.zeros(self.Nx)

    # ---- delay embedding helpers ----
    def _make_delay_indices(self, T):
        """For each step t, indices [t, t-tau, t-2tau, ...] clipped to 0."""
        idx = np.zeros((T, self.n_delays), dtype=int)
        for k in range(self.n_delays):
            shift = k * self.tau
            for t in range(T):
                idx[t, k] = max(t - shift, 0)
        return idx

    # ---- EMA target for slow readout ----
    def _ema(self, sig, tau):
        a = 1.0 / tau
        out = np.zeros_like(sig)
        out[0] = sig[0]
        for t in range(1, len(sig)):
            out[t] = a * sig[t] + (1 - a) * out[t - 1]
        return out

    # ---- training ----
    def fit(self, U, Y, washout):
        """
        U: (T, 1)   teacher input
        Y: (T, 1)   teacher target = U shifted by one step
        """
        T = U.shape[0]
        U_flat = U.ravel()
        Y_flat = Y.ravel()

        # Delay-embedded inputs
        delay_idx = self._make_delay_indices(T)
        U_delayed = U_flat[delay_idx]  # shape (T, n_delays)

        # Slow target: EMA of Y
        Y_slow = self._ema(Y_flat, self.tau_slow)

        # Scheduled noise on inputs only
        noise = np.linspace(self.noise_hi, self.noise_lo, T)
        U_noisy = U_delayed + noise[:, None] * self.rng.standard_normal(
            U_delayed.shape)

        # Teacher-forced drive, collect features
        feat_dim = 2 * self.Nx
        Phi = np.zeros((T, feat_dim))
        self.reset()
        y_fast_prev = np.zeros(1)
        y_slow_prev = np.zeros(1)
        for t in range(T):
            self._step(U_noisy[t], y_fast_prev, y_slow_prev)
            Phi[t] = self._features()
            # Teacher-forcing: feedback is truth (also with a little noise
            # to match closed-loop regime)
            y_fast_prev = np.array([Y_flat[t] + noise[t] *
                                    self.rng.standard_normal()])
            y_slow_prev = np.array([Y_slow[t]])

        Phi_use = Phi[washout:]
        # Two-column target: [fast, slow]
        Y_target = np.column_stack([Y_flat[washout:], Y_slow[washout:]])

        # Spike-weighted ridge: up-weight samples near true spikes
        weights = np.ones(T - washout)
        spikes = spike_indices(Y_flat[washout:], threshold=0.0)
        w_win = self.spike_window
        for p in spikes:
            lo = max(0, p - w_win)
            hi = min(len(weights), p + w_win + 1)
            weights[lo:hi] = self.spike_weight

        # Weighted ridge: solve (Phi^T diag(w) Phi + beta I) W = Phi^T diag(w) Y
        Wmat = np.sqrt(weights)[:, None]
        Phi_w = Phi_use * Wmat
        Y_w = Y_target * Wmat
        A = Phi_w.T @ Phi_w + self.ridge * np.eye(feat_dim)
        B = Phi_w.T @ Y_w
        self.W_out = np.linalg.solve(A, B).T  # shape (2, feat_dim)

        return self

    # ---- synchronization (teacher-forced, no noise) ----
    def synchronize(self, U, Y):
        T = U.shape[0]
        U_flat = U.ravel()
        Y_flat = Y.ravel()
        delay_idx = self._make_delay_indices(T)
        U_delayed = U_flat[delay_idx]
        Y_slow = self._ema(Y_flat, self.tau_slow)

        self.reset()
        y_fast_prev = np.zeros(1)
        y_slow_prev = np.zeros(1)
        for t in range(T):
            self._step(U_delayed[t], y_fast_prev, y_slow_prev)
            y_fast_prev = np.array([Y_flat[t]])
            y_slow_prev = np.array([Y_slow[t]])

        # Return state needed to continue: last feedbacks + history buffer
        # of the last (n_delays * tau) inputs for delay indexing.
        history = deque(U_flat[-self.n_delays * self.tau:].tolist(),
                        maxlen=self.n_delays * self.tau)
        return y_fast_prev, y_slow_prev, history, Y_slow[-1]

    # ---- closed-loop generation with hybrid fade ----
    def closed_loop(self, n_steps, y_fast0, y_slow0, history, slow_init,
                    Y_true_fade=None, fade_steps=50):
        """
        n_steps: total rollout length
        y_fast0, y_slow0: last sync feedbacks
        history: deque of raw input values (length n_delays*tau)
        slow_init: initial EMA state for running EMA of predictions
        Y_true_fade: (fade_steps,) truth values for hybrid-forcing fade
        """
        preds_fast = np.zeros(n_steps)
        preds_slow = np.zeros(n_steps)
        y_fast = np.atleast_1d(y_fast0).astype(float).copy()
        y_slow = np.atleast_1d(y_slow0).astype(float).copy()

        # Running EMA of the network's own fast output (for slow feedback
        # computation by consistency -- but we use the model's own slow
        # readout output, which is already predicted).
        ema_a = 1.0 / self.tau_slow

        for t in range(n_steps):
            # Build current delayed input from history
            hist = list(history)
            # Most recent first: current input is hist[-1]; delays sample
            # back by k*tau.  history holds raw inputs in chronological order.
            u_delayed = np.zeros(self.n_delays)
            H = len(hist)
            for k in range(self.n_delays):
                shift = k * self.tau
                u_delayed[k] = hist[H - 1 - shift] if H - 1 - shift >= 0 \
                    else hist[0]

            # Step the reservoir
            self._step(u_delayed, y_fast, y_slow)

            # Readout: y_hat = W_out @ [x; x^2]
            feats = self._features()
            y_hat = self.W_out @ feats  # shape (2,)
            y_fast_hat = y_hat[0]
            y_slow_hat = y_hat[1]
            preds_fast[t] = y_fast_hat
            preds_slow[t] = y_slow_hat

            # Hybrid-forcing fade on fast feedback for first fade_steps
            if Y_true_fade is not None and t < fade_steps:
                lam = (t + 1) / fade_steps
                y_fast_used = (1 - lam) * Y_true_fade[t] + lam * y_fast_hat
                new_input = y_fast_used
            else:
                y_fast_used = y_fast_hat
                new_input = y_fast_hat

            # Update next-step feedbacks
            y_fast = np.array([y_fast_used])
            y_slow = np.array([y_slow_hat])

            # Push new input into history
            history.append(new_input)

            # Stability guard
            if not np.isfinite(y_fast_hat) or abs(y_fast_hat) > 1e6:
                preds_fast[t + 1:] = np.nan
                preds_slow[t + 1:] = np.nan
                break

        return preds_fast, preds_slow


# ==========================================================
# METRICS
# ==========================================================
def short_horizon_nrmse(y_pred, y_true, horizon=50):
    h = min(horizon, len(y_true))
    yp = np.asarray(y_pred[:h]).ravel()
    yt = np.asarray(y_true[:h]).ravel()
    if not np.all(np.isfinite(yp)):
        return np.inf
    rmse = np.sqrt(np.mean((yp - yt) ** 2))
    denom = np.std(yt)
    return rmse / denom if denom > 0 else np.inf


def composite_score(y_pred, y_true, horizon=50, spike_thresh=0.0):
    """NRMSE_short + 0.5 * |log(n_pred_spikes / n_true_spikes)|."""
    nrmse = short_horizon_nrmse(y_pred, y_true, horizon=horizon)
    if not np.isfinite(nrmse):
        return np.inf
    n_true = count_spikes(y_true, threshold=spike_thresh)
    n_pred = count_spikes(y_pred, threshold=spike_thresh)
    if n_true == 0 or n_pred == 0:
        spike_pen = 5.0
    else:
        spike_pen = abs(np.log(n_pred / n_true))
    return nrmse + 0.5 * spike_pen


# ==========================================================
# LOAD USER'S DATA
# ==========================================================
dataset = np.loadtxt('../../../data/chaotic_data/rulkov_map.csv', delimiter=',')
if dataset.ndim == 2:
    dataset = dataset[:, 0]
data = dataset.reshape(-1, 1)
print(f"Loaded: N={data.shape[0]}, range=[{data.min():.3f}, {data.max():.3f}], "
      f"std={data.std():.3f}")

# Assumes at least ~30000 points; adapt split to actual length
N = data.shape[0]
if N < 24000:
    # Fallback smaller split
    TRAIN_LEN = min(8000, N - 3000)
    GAP = 1000
    TEST_LEN = 2000
    TEST_START = TRAIN_LEN + GAP
else:
    TRAIN_LEN = 20000
    GAP = 2000
    TEST_LEN = 2000
    TEST_START = TRAIN_LEN + GAP  # = 22000

SYNC_LEN = 500
PRED_LEN = TEST_LEN - SYNC_LEN  # 1500 when TEST_LEN=2000

print(f"Train=[0:{TRAIN_LEN}], gap={GAP}, "
      f"test=[{TEST_START}:{TEST_START+TEST_LEN}], "
      f"sync={SYNC_LEN}, pred={PRED_LEN}")

# Training pairs
X_train_raw = data[:TRAIN_LEN - 1]
Y_train_raw = data[1:TRAIN_LEN]
# Test pairs
X_test_raw = data[TEST_START:TEST_START + TEST_LEN - 1]
Y_test_raw = data[TEST_START + 1:TEST_START + TEST_LEN]

scaler = fit_scaler(X_train_raw, method="zscore")
X_train = transform_array(X_train_raw, scaler)
Y_train = transform_array(Y_train_raw, scaler)
X_test = transform_array(X_test_raw, scaler)
Y_test = transform_array(Y_test_raw, scaler)

PRED_LEN = len(Y_test) - SYNC_LEN  # redefine from actual array


# ==========================================================
# PARAMETER GRID
# ==========================================================
# ---- TOP PRIORITY: alpha_min ---------------------------------------------
# Log-spaced across three decades with extra density near paper's optimum.
alpha_min_grid = sorted(set(np.round(np.concatenate([
    np.logspace(-3.0, 0.0, 10),
    [10 ** (-6 / 9)],
    [10 ** (-0.5), 10 ** (-0.7), 10 ** (-0.9)],
]), 5).tolist()))

# ---- HIGH PRIORITY: delay parameters -------------------------------------
# tau * (n_delays - 1) should roughly span one ISI (~10-20 steps for Rulkov).
tau_grid = [3, 5, 8]
n_delays_grid = [2, 3]

# ---- HIGH PRIORITY: feedback scalings -----------------------------------
# Fast feedback is the main closed-loop drive.  Slow feedback provides
# the learned EMA surrogate of the missing y variable.
fb_fast_grid = [0.6, 1.0, 1.3]
fb_slow_grid = [0.3, 0.6, 1.0]

# ---- MEDIUM PRIORITY: spectral radius -----------------------------------
sr_grid = [0.90, 1.00, 1.10]

# ---- MEDIUM PRIORITY: tau_slow (EMA timescale) --------------------------
tau_slow_grid = [50, 150]

# ---- MEDIUM PRIORITY: ridge ---------------------------------------------
ridge_grid = [1e-4, 1e-3]

# ---- LOWER PRIORITY: input scaling --------------------------------------
in_grid = [0.8]   # fixed -- delays carry the phase info now

# ---- FIXED --------------------------------------------------------------
N_RESERVOIR = 1000     # larger reservoir per user's other-model protocol
DENSITY = 0.1
ALPHA_MAX = 1.0
TRAIN_WASHOUT = 500
NOISE_HI = 0.01
NOISE_LO = 0.001
SPIKE_WEIGHT = 4.0
SPIKE_WINDOW = 2
FADE_STEPS = 50

# ---- AVERAGING AXIS: seeds ----------------------------------------------
seed_grid = [42, 7, 2024]

keys = ["alpha_min", "tau", "n_delays", "fb_fast", "fb_slow",
        "sr", "tau_slow", "ridge", "in_scale", "seed"]
grids = [alpha_min_grid, tau_grid, n_delays_grid, fb_fast_grid, fb_slow_grid,
         sr_grid, tau_slow_grid, ridge_grid, in_grid, seed_grid]
combos = list(itertools.product(*grids))

print(f"\nGrid sizes:")
print(f"  alpha_min      : {len(alpha_min_grid)}  (TOP)")
print(f"  tau            : {len(tau_grid)}        (HIGH)")
print(f"  n_delays       : {len(n_delays_grid)}   (HIGH)")
print(f"  fb_fast (zeta) : {len(fb_fast_grid)}    (HIGH)")
print(f"  fb_slow        : {len(fb_slow_grid)}    (HIGH)")
print(f"  sr (rho)       : {len(sr_grid)}         (MED)")
print(f"  tau_slow       : {len(tau_slow_grid)}   (MED)")
print(f"  ridge          : {len(ridge_grid)}      (MED)")
print(f"  in_scale       : {len(in_grid)}         (fixed)")
print(f"  seed           : {len(seed_grid)}       (averaging)")
print(f"  TOTAL          : {len(combos)}")


# ==========================================================
# EVALUATION
# ==========================================================
def evaluate(alpha_min, tau, n_delays, fb_fast, fb_slow,
             sr, tau_slow, ridge, in_scale, seed):
    try:
        esn = DTSESN(
            n_reservoir=N_RESERVOIR,
            alpha_min=alpha_min, alpha_max=ALPHA_MAX,
            spectral_radius=sr,
            input_scaling=in_scale,
            fb_scaling=fb_fast, fb_scaling_slow=fb_slow,
            density=DENSITY, ridge=ridge,
            tau=tau, n_delays=n_delays, tau_slow=tau_slow,
            noise_hi=NOISE_HI, noise_lo=NOISE_LO,
            spike_weight=SPIKE_WEIGHT, spike_window=SPIKE_WINDOW,
            seed=seed,
        )
        esn.fit(X_train, Y_train, washout=TRAIN_WASHOUT)

        y_fast_last, y_slow_last, history, _ = esn.synchronize(
            X_test[:SYNC_LEN], Y_test[:SYNC_LEN])

        # Truth segment used by fade
        Y_true_fade = Y_test[SYNC_LEN:SYNC_LEN + FADE_STEPS].ravel()
        y_pred, _ = esn.closed_loop(
            PRED_LEN, y_fast_last, y_slow_last, history,
            slow_init=y_slow_last[0],
            Y_true_fade=Y_true_fade, fade_steps=FADE_STEPS,
        )
        y_true = Y_test[SYNC_LEN:SYNC_LEN + PRED_LEN].ravel()

        if not np.all(np.isfinite(y_pred)) or np.max(np.abs(y_pred)) > 1e6:
            return {"score": np.inf, "nrmse": np.inf,
                    "n_pred": 0, "n_true": count_spikes(y_true)}

        score = composite_score(y_pred, y_true, horizon=50, spike_thresh=0.0)
        nrmse = short_horizon_nrmse(y_pred, y_true, horizon=50)
        n_pred = count_spikes(y_pred)
        n_true = count_spikes(y_true)
        return {"score": score, "nrmse": nrmse,
                "n_pred": n_pred, "n_true": n_true}
    except Exception:
        return {"score": np.inf, "nrmse": np.inf, "n_pred": 0, "n_true": 0}


# ==========================================================
# RUN GRID
# ==========================================================
print("\n" + "=" * 70)
print("RUNNING GRID SEARCH")
print("=" * 70)
results = []
t0 = time.time()
for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    metrics = evaluate(**params)
    results.append({**params, **metrics})
    if (i + 1) % 300 == 0 or (i + 1) == len(combos):
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (len(combos) - i - 1)
        print(f"  [{i+1}/{len(combos)}]  elapsed={elapsed:.0f}s  ETA={eta:.0f}s")
total_time = time.time() - t0
print(f"\nTotal: {total_time:.1f}s ({total_time/len(combos):.3f}s/trial)")


# ==========================================================
# AGGREGATE OVER SEEDS -- reject any config with a diverged seed
# ==========================================================
group_keys = [k for k in keys if k != "seed"]
grouped = defaultdict(list)
for r in results:
    grouped[tuple(r[k] for k in group_keys)].append(r)

aggregated = []
for key, runs in grouped.items():
    scores = np.array([r["score"] for r in runs])
    nrmses = np.array([r["nrmse"] for r in runs])
    n_preds = np.array([r["n_pred"] for r in runs])
    n_trues = np.array([r["n_true"] for r in runs])
    # Reject if any seed diverged
    if not np.all(np.isfinite(scores)):
        continue
    aggregated.append({
        **dict(zip(group_keys, key)),
        "score_mean": scores.mean(),
        "score_std":  scores.std(),
        "nrmse_mean": nrmses.mean(),
        "n_pred_mean": n_preds.mean(),
        "n_true_mean": n_trues.mean(),
    })

aggregated_sorted = sorted(aggregated, key=lambda r: r["score_mean"])

print("\n" + "=" * 120)
print(f"TOP 25 CONFIGS  (ranked by mean composite score over "
      f"{len(seed_grid)} seeds; divergent seeds reject the config)")
print("=" * 120)
print(f"{'rank':>4}  {'a_min':>7}  {'tau':>3} {'nd':>2}  "
      f"{'zf':>4} {'zs':>4}  {'sr':>4}  {'ts':>4}  {'ridge':>7}  "
      f"{'score':>7}  {'NRMSE':>7}  {'n_pred':>6}/{'n_tru':<5}")
print("-" * 120)
for rank, r in enumerate(aggregated_sorted[:25], 1):
    print(f"{rank:4d}  {r['alpha_min']:7.4f}  "
          f"{int(r['tau']):3d} {int(r['n_delays']):2d}  "
          f"{r['fb_fast']:4.1f} {r['fb_slow']:4.1f}  "
          f"{r['sr']:4.2f}  {int(r['tau_slow']):4d}  "
          f"{r['ridge']:7.1e}  "
          f"{r['score_mean']:7.4f}  {r['nrmse_mean']:7.4f}  "
          f"{r['n_pred_mean']:6.1f}/{r['n_true_mean']:<5.0f}")

if not aggregated:
    print("\nNO CONFIGS SURVIVED ALL SEEDS.  Loosen noise/sigma or check data.")
    sys.exit(0)

best = aggregated_sorted[0]
print(f"\nBEST CONFIG:")
for k in group_keys:
    print(f"  {k:10s} = {best[k]}")
print(f"  score = {best['score_mean']:.4f} +/- {best['score_std']:.4f}")
print(f"  NRMSE = {best['nrmse_mean']:.4f}")


# ==========================================================
# REBUILD BEST MODEL -- median-score seed
# ==========================================================
best_runs = [r for r in results
             if all(r[k] == best[k] for k in group_keys)]
best_runs_sorted = sorted(best_runs, key=lambda r: r["score"])
chosen = best_runs_sorted[len(best_runs_sorted) // 2]
print(f"\nRebuilding with median-score seed = {chosen['seed']} "
      f"(score = {chosen['score']:.4f})")

final = DTSESN(
    n_reservoir=N_RESERVOIR,
    alpha_min=best["alpha_min"], alpha_max=ALPHA_MAX,
    spectral_radius=best["sr"],
    input_scaling=best["in_scale"],
    fb_scaling=best["fb_fast"], fb_scaling_slow=best["fb_slow"],
    density=DENSITY, ridge=best["ridge"],
    tau=best["tau"], n_delays=best["n_delays"], tau_slow=best["tau_slow"],
    noise_hi=NOISE_HI, noise_lo=NOISE_LO,
    spike_weight=SPIKE_WEIGHT, spike_window=SPIKE_WINDOW,
    seed=chosen["seed"],
)
final.fit(X_train, Y_train, washout=TRAIN_WASHOUT)
y_fast_last, y_slow_last, history, _ = final.synchronize(
    X_test[:SYNC_LEN], Y_test[:SYNC_LEN])
Y_true_fade = Y_test[SYNC_LEN:SYNC_LEN + FADE_STEPS].ravel()
y_pred_z, y_slow_z = final.closed_loop(
    PRED_LEN, y_fast_last, y_slow_last, history,
    slow_init=y_slow_last[0],
    Y_true_fade=Y_true_fade, fade_steps=FADE_STEPS,
)
y_true_z = Y_test[SYNC_LEN:SYNC_LEN + PRED_LEN].ravel()

y_pred_raw = inverse_transform_array(y_pred_z.reshape(-1, 1), scaler).ravel()
y_true_raw = inverse_transform_array(y_true_z.reshape(-1, 1), scaler).ravel()

nrmse_final = short_horizon_nrmse(y_pred_z, y_true_z, horizon=50)
score_final = composite_score(y_pred_z, y_true_z, horizon=50)
n_pred_final = count_spikes(y_pred_z)
n_true_final = count_spikes(y_true_z)
print(f"\nFinal metrics:")
print(f"  Composite score       : {score_final:.4f}")
print(f"  Short-horizon NRMSE   : {nrmse_final:.4f}  (first 50 steps)")
print(f"  Spikes: pred={n_pred_final}, true={n_true_final}")


# ==========================================================
# PLOT -- single prediction overlay only
# ==========================================================
fig, ax = plt.subplots(figsize=(14, 4))
t_axis = np.arange(PRED_LEN)
ax.plot(t_axis, y_true_raw, color="black", lw=0.9, label="true $x_t$")
ax.plot(t_axis, y_pred_raw, color="crimson", lw=0.9, ls="--",
        label="DTS-ESN closed-loop")
ax.set_xlabel("step (after sync)")
ax.set_ylabel("$x_t$")
ax.set_title(
    f"Enhanced DTS-ESN closed-loop Rulkov  |  "
    f"score={score_final:.3f}, NRMSE@50={nrmse_final:.3f}, "
    f"spikes pred/true={n_pred_final}/{n_true_final}  |  "
    r"$\alpha_{\min}$="f"{best['alpha_min']:.3f}, "
    f"tau={best['tau']}/nd={best['n_delays']}, "
    r"$\zeta_f$="f"{best['fb_fast']}, "
    r"$\zeta_s$="f"{best['fb_slow']}, "
    r"$\rho$="f"{best['sr']}"
)
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

Loaded: N=30000, range=[-2.239, 1.366], std=1.019
Train=[0:20000], gap=2000, test=[22000:24000], sync=500, pred=1500

Grid sizes:
  alpha_min      : 13  (TOP)
  tau            : 3        (HIGH)
  n_delays       : 2   (HIGH)
  fb_fast (zeta) : 3    (HIGH)
  fb_slow        : 3    (HIGH)
  sr (rho)       : 3         (MED)
  tau_slow       : 2   (MED)
  ridge          : 2      (MED)
  in_scale       : 1         (fixed)
  seed           : 3       (averaging)
  TOTAL          : 25272

RUNNING GRID SEARCH


KeyboardInterrupt: 